<a href="https://colab.research.google.com/github/awadhmohmmed/awadh-alhajri/blob/main/lap_final_1_2_3_4_5_6_7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# =====================================================

# SMART CONSTRUCTION PROJECTS ASSISTANT

# Unified Project - Labs 1 to 7

# Company: Jazirat Socotra Contracting

# =====================================================

import re

import sqlite3

import hashlib

from dataclasses import dataclass

from concurrent.futures import ThreadPoolExecutor

from pydantic import BaseModel, Field, ValidationError

# =====================================================

# LAB 1 & 2: ASSISTANT CONFIGURATION

# =====================================================

ASSISTANT_NAME = "Smart Construction Projects Assistant"

COMPANY_NAME = "Jazirat Socotra Contracting"

SYSTEM_PROMPT = """

You are an intelligent construction project assistant.

Your responsibilities:

1. Manage construction projects.

2. Monitor excavation and backfilling.

3. Track equipment and trucks.

4. Calculate quantities and costs.

5. Monitor project progress.

6. Answer using available company information.

"""

print("Assistant:", ASSISTANT_NAME)

# =====================================================

# LAB 3: PYDANTIC SCHEMA

# =====================================================

class ConstructionProject(BaseModel):

    project_name: str = Field(min_length=2)

    location: str

    activity: str

    quantity: float = Field(ge=0)

    unit: str

    progress: float = Field(ge=0, le=100)

# Sample Project

project = ConstructionProject(

    project_name="Road Construction",

    location="Saudi Arabia",

    activity="Excavation",

    quantity=400000,

    unit="m3",

    progress=75

)

# =====================================================

# LAB 3: COMPANY DATA

# =====================================================

projects = {

    "P001": {

        "name": "Road Construction",

        "activity": "Excavation",

        "quantity": 400000,

        "progress": 75

    },

    "P002": {

        "name": "Earthworks",

        "activity": "Backfilling",

        "quantity": 300000,

        "progress": 60

    }

}

equipment = {

    "Excavators": 20,

    "Loaders": 15,

    "Dump Trucks": 100,

    "Bulldozers": 5,

    "Mobile Crushers": 2,

    "Graders": 4

}

# =====================================================

# LAB 3: TOOL REGISTRY

# =====================================================

def check_project():

    return projects

def list_equipment():

    return equipment

def calculate_cost(quantity, unit_cost):

    return quantity * unit_cost

TOOL_REGISTRY = {

    "check_project": check_project,

    "list_equipment": list_equipment,

    "calculate_cost": calculate_cost

}

# Execute Tool

def execute_tool(tool_name, **kwargs):

    if tool_name not in TOOL_REGISTRY:

        return "Unknown Tool"

    return TOOL_REGISTRY[tool_name](**kwargs)

# Parallel Tool Calls

def parallel_tools():

    with ThreadPoolExecutor(max_workers=3) as executor:

        results = list(executor.map(

            lambda name: execute_tool(name),

            ["check_project", "list_equipment"]

        ))

    return results

# =====================================================

# LAB 4: DOCUMENT PROCESSING

# =====================================================

project_text = """

Excavation: Earth removal and site preparation.

Backfilling: Filling and compacting excavated areas.

Crushing: Processing excavated materials.

Transportation: Moving materials using dump trucks.

Safety: Monitoring workers and equipment.

Energy: Installation of electrical infrastructure.

"""

# Clean Documents

clean_text = re.sub(

    r"\s+",

    " ",

    project_text

).strip()

@dataclass

class Document:

    source: str

    content: str

document = Document(

    source="company_projects",

    content=clean_text

)

# Recursive Text Chunking

def split_text(text, chunk_size=100, overlap=20):

chunks = []

    start = 0

    while start < len(text):

        end = min(start + chunk_size, len(text))

        chunks.append(text[start:end])

        if end == len(text):

            break

        start = end - overlap

    return chunks

recursive_chunks = split_text(document.content)

# =====================================================

# LAB 5: KNOWLEDGE RETRIEVAL

# =====================================================

def retrieve_information(question, chunks):

    question_words = set(

        re.findall(r"\w+", question.lower())

    )

    scored = []

    for chunk in chunks:

        words = set(

            re.findall(r"\w+", chunk.lower())

        )

        score = len(question_words.intersection(words))

        scored.append((score, chunk))

    scored.sort(reverse=True)

    return [

        chunk for score, chunk in scored[:3]

        if score > 0

    ]

# =====================================================

# LAB 6: SQLITE CHAT MEMORY

# =====================================================

conn = sqlite3.connect(

    "construction.db",

    check_same_thread=False

)

conn.execute("""

CREATE TABLE IF NOT EXISTS chat (

    question TEXT,

    answer TEXT

)

""")

conn.commit()

def save_conversation(question, answer):

    conn.execute(

        "INSERT INTO chat VALUES (?, ?)",

        (question, answer)

    )

    conn.commit()

def retrieve_memory(limit=3):

    return conn.execute(

        "SELECT question, answer FROM chat "

        "ORDER BY rowid DESC LIMIT ?",

        (limit,)

    ).fetchall()

# Sliding Window

def sliding_window():

    return retrieve_memory(3)

# Recursive Summarisation

def summarize_memory():

    conversations = retrieve_memory(10)

    summary = []

    for question, answer in reversed(conversations):

        summary.append(

            f"Question: {question} | Answer: {answer}"

        )

    return "\n".join(summary)

# Fidelity Gate

def fidelity_gate(summary):

    important_words = [

        "excavation",

        "backfilling",

        "trucks",

        "crushing",

        "quantity",

        "cost"

    ]

    return {

        word: word in summary.lower()

        for word in important_words

    }

# Hybrid Compactor

def hybrid_compactor():

    recent = sliding_window()

    summary = summarize_memory()

    return {

        "recent_messages": recent,

        "summary": summary,

        "fidelity": fidelity_gate(summary)

    }

# =====================================================

# LAB 7: LOOP DETECTOR

# =====================================================

class LoopDetector:

    def _init_(self, max_repeats=2):

        self.max_repeats = max_repeats

        self.history = {}

    def check(self, question):

        key = hashlib.md5(

            question.lower().strip().encode()

        ).hexdigest()

        self.history[key] = (

            self.history.get(key, 0) + 1

        )

        if self.history[key] > self.max_repeats:

            return False

        return True
        loop_detector = LoopDetector()

# =====================================================

# LAB 7: EXECUTION TRACE

# =====================================================

def observe(function):

    def wrapper(*args, **kwargs):

        print("TRACE:", function._name_)

        result = function(*args, **kwargs)

        print("TRACE COMPLETED")

        return result

    return wrapper

# =====================================================

# INTEGRATION: MAIN ASSISTANT

# =====================================================

@observe

def construction_assistant(question):

    # 1. Prevent Infinite Loops

    if not loop_detector.check(question):

        return "Repeated request detected. Execution stopped."

    # 2. Search Company Documents

    knowledge = retrieve_information(

        question,

        recursive_chunks

    )

    # 3. Retrieve Project Data

    if "equipment" in question.lower():

        result = execute_tool("list_equipment")

    elif "cost" in question.lower():

        result = execute_tool(

            "calculate_cost",

            quantity=400000,

            unit_cost=9

        )

    elif "project" in question.lower():

        result = execute_tool("check_project")

    else:

        result = "General construction inquiry"

    # 4. Build Answer

    answer = f"""

    Assistant: {ASSISTANT_NAME}

    Project Information:

    {result}

    Knowledge:

    {knowledge}

    """

    # 5. Save Conversation

    save_conversation(question, answer)

    return answer

# =====================================================

# EVALUATION

# =====================================================

def evaluate_answer(answer):

    if answer and len(answer.strip()) > 10:

        return "PASS"

    return "FAIL"

# =====================================================

# FINAL TEST

# =====================================================

question = "Show construction project equipment"

answer = construction_assistant(question)

print("\nFINAL ANSWER")

print(answer)

print("\nEVALUATION")

print(evaluate_answer(answer))

print("\nCHAT MEMORY")

print(hybrid_compactor())

print("\nPROJECT COMPLETED")